# Classic CF & Metrics


This is the first stage where we fit models and *grade* them. We fit two models: a content-based cosine-rank baseline (no learning, just TF-IDF similarity), and an alternating-least-squares matrix factorization model (the classic recsys workhorse). Together they form the floor that every later model — Two-Tower, ranker, sequence model, LLM re-ranker — has to beat to justify its extra complexity.

Just as important: this notebook introduces the metrics. **Recall@K and NDCG@K** measure ranking quality on held-out items; **Coverage@K and Novelty@K** measure beyond-accuracy properties (does the model explore the catalog, does it confine itself to the popular head). The `Metricator` class is the single object every later notebook uses to compare models — and every A/B test in REC:08 reads its output.


## Setup


In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline

backend_inline.set_matplotlib_formats("svg")
plt.rcParams["figure.dpi"] = 110

from notebooks.recsys.config import MovieLensConfig
from notebooks.recsys.data import load_movielens, time_split
from notebooks.recsys.features import FeatureStore
from notebooks.recsys.metrics import (
    Metricator, recall_at_k, ndcg_at_k, coverage_at_k, novelty_at_k,
)
from notebooks.recsys.models.classic import ALSRecommender, ContentRecommender


## The ALS objective

Matrix factorization models the rating matrix $\mathbf{R} \in \mathbb{R}^{|U|\times|V|}$ as a low-rank product:

$$\mathbf{R} \approx \mathbf{U}\mathbf{V}^{\top}, \qquad \mathbf{U}\in\mathbb{R}^{|U|\times d}, \quad \mathbf{V}\in\mathbb{R}^{|V|\times d}.$$

The squared-error objective plus an $L_2$ penalty on both factor matrices is:

$$\min_{\mathbf{U}, \mathbf{V}} \quad \sum_{(u, i)\in\Omega} \big(r_{ui} - \mathbf{u}_u^{\top}\mathbf{v}_i\big)^2 + \lambda\big(\|\mathbf{u}_u\|^2 + \|\mathbf{v}_i\|^2\big),$$

where $\Omega$ is the set of observed interactions — not every $(u, i)$ pair, just the rows that exist in `ratings`. The problem is non-convex jointly in $(\mathbf{U}, \mathbf{V})$ but **biconvex**: fix $\mathbf{V}$, and the objective in $\mathbf{U}$ is positive-definite quadratic with a closed-form minimizer.

**ALS update.** Fix $\mathbf{V}$ and write the per-user objective as a function of $\mathbf{u}_u$ only:

$$\mathcal{L}_u(\mathbf{u}_u) = \sum_{i\in\Omega_u} (r_{ui} - \mathbf{u}_u^{\top}\mathbf{v}_i)^2 + \lambda\|\mathbf{u}_u\|^2.$$

Setting $\nabla_{\mathbf{u}_u}\mathcal{L}_u = 0$ gives the normal equations:

$$\boxed{\,\mathbf{u}_u \leftarrow \big(\mathbf{V}_u^{\top}\mathbf{V}_u + \lambda\mathbf{I}\big)^{-1}\mathbf{V}_u^{\top}\mathbf{r}_u,\,}$$

where $\mathbf{V}_u$ is the rows of $\mathbf{V}$ restricted to items user $u$ rated and $\mathbf{r}_u$ is the vector of those ratings. The update for $\mathbf{v}_i$ is symmetric. Alternate the two until convergence. This is the original Koren, Bell & Volinsky (2009) formulation.

Implementation lives in `notebooks.recsys.models.classic.ALSRecommender`. The key trick that makes our pure-Numpy version tractable: the expensive part of each epoch is $|U| \times$$ d \times$$ d$ matrix inversions; cache the inverse of $\mathbf{V}^{\top}\mathbf{V} + \lambda\mathbf{I}$ across all user updates (one inverse per epoch).


## Beyond-accuracy metrics

**Recall@K.** Take the user's held-out positive items (rating $\ge 4$) as the *relevant set*. The fraction that appear in the top-K predictions is Recall@K.

**NDCG@K.** Discounted cumulative gain at K, normalized by the ideal DCG:

$$\mathrm{DCG}@K = \sum_{i=1}^{K}\frac{2^{\mathrm{rel}(i)}-1}{\log_2(i+1)}, \qquad \mathrm{NDCG}@K = \frac{\mathrm{DCG}_K}{\mathrm{IDCG}_K}.$$

For binary relevance, $\mathrm{rel}(i) \in \{0, 1\}$, and the formula reduces to $\sum_{i : \mathrm{rel}(i)=1} \frac{1}{\log_2(i+1)} \Big/ \sum_{i=1}^{\min(|\mathcal{R}|, K)} \frac{1}{\log_2(i+1)}$.

**Coverage@K.** Fraction of the catalog that the model ever recommends in its top-K across the evaluation set. Defined over the *predicted* distribution, not the held-out ground truth. A perfect Recall@K model can have Coverage $\approx 0$ if it always picks the same popular items.

**Novelty@K.** Average self-information $-\log_2 p(i)$ of recommended items, where $p(i)$ is the empirical item popularity in the held-out set. Higher is more surprising. Too high (recommending nothing but long-tail unknowns) and the user churns; too low (recommending nothing but blockbusters) and they get bored.

:::{.callout-warning}
NDCG is *graded* — it cares about positions. Recall is *binary* — equal credit for hitting rank 1 and rank K. A model that beats another on NDCG but ties on Recall has better *ranking*; equal NDCG and higher Recall means it surfaces more items, but in worse positions. Use both.
:::


In [ ]:
cfg = MovieLensConfig(name="ml-100k")
ds = load_movielens(cfg)
train, val = time_split(ds.ratings, val_frac=0.2)
print(f"split: train={len(train):,}  val={len(val):,}")
print(f"unique eval users: {val['user_id'].nunique()}")


## Fitting ALS


In [ ]:
als = ALSRecommender(
    n_users=ds.n_users, n_items=ds.n_items,
    d=16, reg=10.0, epochs=8, seed=0,
)
als.fit(train, ds.user_index, ds.item_index, verbose=True)


Residual MSE drops steadily. The constant $\lambda = 10$ we chose is a heavy regularizer appropriate to ml-100k's tiny per-user samples; tune down to $\sim 0.1$ for ml-25m. Note the MSE shrinks below $1.0$ on 5 ratings scale (so RMSE $\approx 0.9$), which corresponds to "predict within about one star" — a reasonable MF baseline on 100k.


In [ ]:
# Look at the top-10 recommendations for one user.
uid = int(val["user_id"].value_counts().idxmax())
rated = set(train[train["user_id"]==uid]["item_id"].tolist())
recs = als.recommend(uid, k=10, exclude=rated)
print(f"Recommendations for user {uid} (already excluded their training items):")
for iid in recs:
    title = ds.movies.loc[ds.movies["item_id"]==iid, "title"].iloc[0]
    print(f"  {iid:6d}  {title}")


## Fitting the content-based baseline


In [ ]:
store = FeatureStore(ds)
cr = ContentRecommender(store=store, training=train)
crecs = cr.recommend(uid, k=10, exclude=rated)
print(f"Content-based recommendations for user {uid}:")
for iid in crecs:
    title = ds.movies.loc[ds.movies["item_id"]==iid, "title"].iloc[0]
    print(f"  {iid:6d}  {title}")


The content-based recommender's top picks for the user are similar in genre but lower-popularity (because of how the cosine rank works — popular items dominate any TF-IDF profile). User-history exclusions still apply.


## Comparing metrics


In [ ]:
metricator = Metricator(val)
als_metrics = metricator.evaluate(als.recommend, k=50)
content_metrics = metricator.evaluate(cr.recommend, k=50)

rows = []
for key in ["recall@50", "ndcg@50", "coverage@50", "novelty@50"]:
    rows.append({"metric": key, "ALS": f"{als_metrics[key]:.4f}",
                 "content": f"{content_metrics[key]:.4f}"})
import pandas as pd
pd.DataFrame(rows)


**Reading the numbers.** ALS is ahead on accuracy (Recall@50 and NDCG@50 are an order of magnitude better) because it learns user-specific preferences — every user's ratings update their embedding and everyone else's. The content-based model has nothing to learn from except TF-IDF similarity; it can recover *genres* but cannot distinguish users who rate within the same genre differently.

Coverage tells the opposite story. ALS covers only $\sim$$ 15\%$ of the items in its top-50; content covers $\sim$$ 67\%$. ALS exploits — it picks from a small set of items whose embeddings happen to dot to high scores. Content explores — its TF-IDF scoring touches the whole catalog. This is exactly why we measure both: narrowly-good recommendations are not the goal. A better demo app (REC:09) will balance them.

Novelty is similar: 4.5 bits vs 5.5. The content model's recommendations are more surprising, but at the cost of accuracy. We will see the same trade-off worsen with deeper models without novelty regularization.


## Why RMSE can mislead

ALS's objective is *rating reconstruction*. A very common textbook finding: minimizing $\sum (r_{ui} - \hat{r}_{ui})^2$ does *not* maximize NDCG. Here's a sketch.

Define the rating task and the ranking task on the same data. The rating-optimal predictor at user $u$ is $\hat{r}_{ui} = \mathbf{u}_u^{\top}\mathbf{v}_i$. To rank items for $u$, sort by $\hat{r}_{ui}$. Now imagine user $u$ has rated 100 items. The top item under $\hat{r}$ might be a 5-star rating that the model nails. But the second-best item under $\hat{r}$ might be a 4-star rating that the model overshot. *Sorting by predicted rating ends up sorting by over-confident items*, and over-confidence is orthogonal to ranking value.

The fix is either (a) retrain models with a listwise loss (REC:05), or (b) use ranking metrics as the *primary* objective plus a small RMSE auxiliary term. Both are standing practice. The point of measuring NDCG here, on a model trained on RMSE, is to make the gap visible.


## Caveats and link forward

- **ALS scales.** The per-epoch cost is $\mathcal{O}(|\Omega|d + (|U| + |V|)d^3)$. With $d=128$ on ml-25m, that's many minutes per epoch on a single thread. Production systems precompute ALS Spark jobs overnight. For the demo app (REC:09), we stick with $d$ and almost everything else small.
- **Negative sampling hint.** ALS predicts ratings — there is no $r_{ui} = 0$ for *would-not-watch* interactions. The Two-Tower model in REC:04 fixes that by treating *all* unobserved pairs as 0 targets and sampling negatives. That change is the entire conceptual leap from "explicit MF" to "modern retrieval."
- **Coverage isn't enough.** A model that randomly samples the catalog has Coverage = 1. Random is bad. We will replace Coverage with a *Gini* on the prediction distribution in REC:08 — a metric that's low both for popularity exploiters and for random recommenders but high only for genuinely *diverse* recommenders.

Next: REC:04 builds the Two-Tower retrieval model — the first deep learning model of the course — and a FAISS index over its item embeddings to serve candidates in real time.
